In [1]:
from pathlib import Path
import json

project_folder = Path(r"D:\electricity-tariff-optimisation")
processed_folder = project_folder / "data" / "processed"
src_folder = project_folder / "src"
src_folder.mkdir(parents=True, exist_ok=True)

required_files = {
    "database": (
        processed_folder / "electricity_analysis.duckdb"
    ),
    "tariff_config": (
        processed_folder / "locked_optional_switch_tariff_2013.json"
    ),
    "assignments": (
        processed_folder / "locked_customer_tariff_assignments_2013.csv"
    ),
    "profile_model": (
        project_folder / "models"
        / "design_profile_kmeans_2013.joblib"
    ),
    "profile_metadata": (
        project_folder / "models"
        / "design_profile_kmeans_2013.json"
    ),
}

for name, file_path in required_files.items():
    assert file_path.is_file(), f"Missing input: {file_path}"
    print(f"PASS: {name}")

tariff_config = json.loads(
    required_files["tariff_config"].read_text(encoding="utf-8")
)

print("\nPipeline inputs ready.")
print("The pipeline will use saved tariff prices and assignments.")
print("No model fitting or price optimisation was performed.")

PASS: database
PASS: tariff_config
PASS: assignments
PASS: profile_model
PASS: profile_metadata

Pipeline inputs ready.
The pipeline will use saved tariff prices and assignments.
No model fitting or price optimisation was performed.


In [2]:
module_code = '''
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


def load_period_consumption(
    database_file,
    assignments_file,
    start_utc,
    end_utc,
):
    """Load and validate a complete UTC half-hour period."""
    database_file = Path(database_file)
    assignments_file = Path(assignments_file)

    if not database_file.is_file():
        raise FileNotFoundError(database_file)
    if not assignments_file.is_file():
        raise FileNotFoundError(assignments_file)

    start = pd.Timestamp(start_utc)
    end = pd.Timestamp(end_utc)

    start = (
        start.tz_localize("UTC")
        if start.tzinfo is None
        else start.tz_convert("UTC")
    )
    end = (
        end.tz_localize("UTC")
        if end.tzinfo is None
        else end.tz_convert("UTC")
    )

    if pd.isna(start) or pd.isna(end) or start >= end:
        raise ValueError("A valid start before end is required.")

    for boundary in (start, end):
        if boundary != boundary.floor("30min"):
            raise ValueError("Period boundaries must align to half-hours.")

    assignments = pd.read_csv(
        assignments_file,
        dtype={"household_id": "string"},
    )
    required_columns = {"household_id", "assigned_tariff"}
    if not required_columns.issubset(assignments.columns):
        raise ValueError("Required assignment columns are missing.")

    if assignments.empty or assignments["household_id"].isna().any():
        raise ValueError("Assignments must contain valid households.")
    if assignments["household_id"].duplicated().any():
        raise ValueError("Duplicate household assignments found.")
    if not assignments["assigned_tariff"].isin(
        ["Flat", "Time-of-use"]
    ).all():
        raise ValueError("Unknown or missing tariff assignment.")

    with duckdb.connect(
        str(database_file), read_only=True
    ) as connection:
        consumption = connection.execute(
            """
            SELECT household_id, interval_start_utc, energy_kwh
            FROM analysis_consumption_2013
            WHERE interval_start_utc >= ?
              AND interval_start_utc < ?
            ORDER BY household_id, interval_start_utc
            """,
            [
                start.tz_localize(None).to_pydatetime(),
                end.tz_localize(None).to_pydatetime(),
            ],
        ).fetchdf()

    consumption["interval_start_utc"] = pd.to_datetime(
        consumption["interval_start_utc"], utc=True
    )

    if set(consumption["household_id"]) != set(
        assignments["household_id"]
    ):
        raise ValueError("Period households differ from saved assignments.")

    if consumption.duplicated(
        ["household_id", "interval_start_utc"]
    ).any():
        raise ValueError("Duplicate household intervals found.")

    energy = consumption["energy_kwh"].to_numpy(dtype=float)
    if not np.isfinite(energy).all() or (energy < 0).any():
        raise ValueError("Energy values must be finite and non-negative.")

    expected_times = pd.date_range(
        start=start,
        end=end,
        freq="30min",
        inclusive="left",
    )
    if not consumption["interval_start_utc"].isin(
        expected_times
    ).all():
        raise ValueError("Unexpected interval timestamps found.")

    counts = consumption.groupby("household_id").size()
    if not counts.eq(len(expected_times)).all():
        raise ValueError("Incomplete household interval coverage.")

    return consumption
'''

module_file = src_folder / "consumption_pipeline.py"
module_file.write_text(
    module_code.lstrip(),
    encoding="utf-8",
)

# Load the module directly from its saved file.
import importlib.util

module_spec = importlib.util.spec_from_file_location(
    "consumption_pipeline",
    module_file,
)
pipeline = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(pipeline)

pipeline_validation = pipeline.load_period_consumption(
    database_file=required_files["database"],
    assignments_file=required_files["assignments"],
    start_utc="2013-07-01",
    end_utc="2014-01-01",
)

print(f"Module saved: {module_file}")
print(
    "Households:",
    pipeline_validation["household_id"].nunique(),
)
print(f"Intervals: {len(pipeline_validation):,}")
print(
    "Energy (kWh): "
    f"{pipeline_validation['energy_kwh'].sum():,.3f}"
)
print("Period data checks: PASS")

Module saved: D:\electricity-tariff-optimisation\src\consumption_pipeline.py
Households: 248
Intervals: 2,190,336
Energy (kWh): 489,289.068
Period data checks: PASS


In [3]:
billing_code = '''

def calculate_period_bills(consumption, assignments_file, config):
    """Calculate energy charges using saved rates and assignments."""
    bands = ["Peak", "Shoulder", "Off-peak"]

    local_start = consumption["interval_start_utc"].dt.tz_convert(
        "Europe/London"
    )
    hour = local_start.dt.hour
    weekday = local_start.dt.dayofweek < 5

    time_band = np.select(
        [
            hour < 7,
            weekday & (hour >= 16) & (hour < 20),
        ],
        ["Off-peak", "Peak"],
        default="Shoulder",
    )

    band_totals = (
        consumption.assign(time_band=time_band)
        .groupby(["household_id", "time_band"])["energy_kwh"]
        .sum()
        .unstack("time_band", fill_value=0)
        .reindex(columns=bands, fill_value=0)
    )

    rates = pd.Series(config["rates_gbp_per_kwh"]).reindex(bands)
    costs = pd.Series(
        config["variable_cost_rates_gbp_per_kwh"]
    ).reindex(bands)
    baseline_rate = float(config["baseline_rate_gbp_per_kwh"])

    rate_values = np.r_[
        rates.to_numpy(dtype=float),
        costs.to_numpy(dtype=float),
        baseline_rate,
    ]
    if not np.isfinite(rate_values).all() or (rate_values < 0).any():
        raise ValueError("Rates must be finite and non-negative.")

    assignments = pd.read_csv(
        assignments_file,
        index_col="household_id",
    )

    if assignments.index.has_duplicates:
        raise ValueError("Duplicate household assignments found.")
    if set(assignments.index) != set(band_totals.index):
        raise ValueError("Assignments and consumption households differ.")
    if not assignments["assigned_tariff"].isin(
        ["Flat", "Time-of-use"]
    ).all():
        raise ValueError("Unknown or missing tariff assignment.")

    results = assignments[["assigned_tariff"]].reindex(
        band_totals.index
    ).copy()

    results["energy_kwh"] = band_totals.sum(axis=1)
    results["baseline_charge_gbp"] = (
        results["energy_kwh"] * baseline_rate
    )
    results["tou_charge_gbp"] = band_totals.mul(
        rates, axis="columns"
    ).sum(axis=1)
    results["assigned_charge_gbp"] = np.where(
        results["assigned_tariff"].eq("Time-of-use"),
        results["tou_charge_gbp"],
        results["baseline_charge_gbp"],
    )
    results["saving_gbp"] = (
        results["baseline_charge_gbp"]
        - results["assigned_charge_gbp"]
    )

    positive_baseline = results["baseline_charge_gbp"].where(
        results["baseline_charge_gbp"] > 0
    )
    results["saving_pct"] = (
        100 * results["saving_gbp"] / positive_baseline
    )
    results["bill_increase_pct"] = -results["saving_pct"]
    results["variable_cost_gbp"] = band_totals.mul(
        costs, axis="columns"
    ).sum(axis=1)

    return band_totals, results
'''

# Replace this function section if the cell is rerun.
existing_code = module_file.read_text(encoding="utf-8")
function_marker = "\ndef calculate_period_bills("
base_code = existing_code.split(function_marker, 1)[0]

module_file.write_text(
    base_code.rstrip() + "\n" + billing_code,
    encoding="utf-8",
)

module_spec = importlib.util.spec_from_file_location(
    "consumption_pipeline",
    module_file,
)
pipeline = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(pipeline)

pipeline_bands, pipeline_bills = pipeline.calculate_period_bills(
    consumption=pipeline_validation,
    assignments_file=required_files["assignments"],
    config=tariff_config,
)

# Compare every household against the previously saved results.
import pandas as pd
import numpy as np

saved_bills = pd.read_csv(
    processed_folder / "validation_locked_tariff_customer_impact_2013.csv",
    index_col="household_id",
)

assert not saved_bills.index.has_duplicates
assert set(saved_bills.index) == set(pipeline_bills.index)
saved_bills = saved_bills.reindex(pipeline_bills.index)

assert saved_bills["assigned_tariff"].equals(
    pipeline_bills["assigned_tariff"]
), "Saved tariff assignments differ."

check_columns = [
    "energy_kwh",
    "baseline_charge_gbp",
    "tou_charge_gbp",
    "assigned_charge_gbp",
    "saving_gbp",
    "variable_cost_gbp",
]

differences = (
    pipeline_bills[check_columns] - saved_bills[check_columns]
).abs()

assert np.isfinite(differences.to_numpy()).all()
assert differences.le(1e-6).all().all(), \
    "Pipeline results differ from saved results."

revenue = pipeline_bills["assigned_charge_gbp"].sum()
variable_cost = pipeline_bills["variable_cost_gbp"].sum()

print("Household-level saved-result reconciliation: PASS")
print(
    "Total customer saving (GBP): "
    f"{pipeline_bills['saving_gbp'].sum():,.4f}"
)
print(
    "Contribution margin (%): "
    f"{100 * (revenue - variable_cost) / revenue:.4f}"
)
print(
    "Maximum household increase (%): "
    f"{pipeline_bills['bill_increase_pct'].max():.4f}"
)
print("Results are before rebates.")

Household-level saved-result reconciliation: PASS
Total customer saving (GBP): 3,925.4984
Contribution margin (%): 25.4851
Maximum household increase (%): 2.7651
Results are before rebates.


In [4]:
profile_code = '''

def predict_period_profiles(consumption, model_file, metadata_file):
    """Apply a trusted saved model to complete period profiles."""
    import json
    import joblib

    metadata = json.loads(
        Path(metadata_file).read_text(encoding="utf-8")
    )

    local_start = consumption["interval_start_utc"].dt.tz_convert(
        "Europe/London"
    )

    profile_data = consumption.assign(
        day_type=np.where(
            local_start.dt.dayofweek < 5,
            "Weekday",
            "Weekend",
        ),
        slot=local_start.dt.hour * 2 + local_start.dt.minute // 30,
    )

    expected_columns = pd.MultiIndex.from_product(
        [["Weekday", "Weekend"], range(48)],
        names=["day_type", "slot"],
    )

    profiles = (
        profile_data
        .groupby(["household_id", "day_type", "slot"])["energy_kwh"]
        .mean()
        .unstack(["day_type", "slot"])
        .reindex(columns=expected_columns)
        .sort_index()
    )

    values = profiles.to_numpy(dtype=float)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Incomplete or invalid consumption profiles.")

    profile_means = profiles.mean(axis=1)
    if not profile_means.gt(0).all():
        raise ValueError("Zero-consumption profiles cannot be normalised.")

    normalised = profiles.div(profile_means, axis=0)
    normalised.columns = [
        f"{day_type.lower()}_{slot:02d}"
        for day_type, slot in expected_columns
    ]

    if list(normalised.columns) != metadata["feature_columns"]:
        raise ValueError("Features differ from saved model metadata.")

    # Load only trusted model files created in this project.
    model = joblib.load(model_file)
    if model.n_features_in_ != normalised.shape[1]:
        raise ValueError("Model feature count does not match profiles.")

    predictions = model.predict(normalised.to_numpy())

    labels = {
        int(cluster_id): label
        for cluster_id, label in metadata["profile_labels"].items()
    }
    results = pd.DataFrame(
        {"profile_cluster": predictions},
        index=normalised.index,
    )
    results["profile_group"] = results["profile_cluster"].map(labels)

    if results["profile_group"].isna().any():
        raise ValueError("Predicted cluster has no saved profile label.")

    return normalised, results
'''

# Replace this function section if the cell is rerun.
existing_code = module_file.read_text(encoding="utf-8")
function_marker = "\ndef predict_period_profiles("
base_code = existing_code.split(function_marker, 1)[0]

module_file.write_text(
    base_code.rstrip() + "\n" + profile_code,
    encoding="utf-8",
)

module_spec = importlib.util.spec_from_file_location(
    "consumption_pipeline",
    module_file,
)
pipeline = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(pipeline)

pipeline_profiles, pipeline_groups = pipeline.predict_period_profiles(
    consumption=pipeline_validation,
    model_file=required_files["profile_model"],
    metadata_file=required_files["profile_metadata"],
)

saved_comparison = pd.read_csv(
    processed_folder / "customer_profile_h1_h2_comparison_2013.csv",
    index_col="household_id",
)
saved_features = pd.read_csv(
    processed_folder / "customer_validation_normalised_profiles_2013.csv",
    index_col="household_id",
)

assert not saved_comparison.index.has_duplicates
assert not saved_features.index.has_duplicates
assert set(pipeline_groups.index) == set(saved_comparison.index)
assert set(pipeline_profiles.index) == set(saved_features.index)
assert list(pipeline_profiles.columns) == list(saved_features.columns)

saved_comparison = saved_comparison.reindex(pipeline_groups.index)
saved_features = saved_features.reindex(pipeline_profiles.index)

assert np.allclose(
    pipeline_profiles.to_numpy(),
    saved_features.to_numpy(),
    rtol=0,
    atol=1e-10,
), "Profile features differ from saved validation features."

assert np.array_equal(
    pipeline_groups["profile_cluster"].to_numpy(),
    saved_comparison["validation_cluster"].to_numpy(),
), "Predictions differ from saved validation clusters."

assert np.array_equal(
    pipeline_groups["profile_group"].to_numpy(),
    saved_comparison["validation_profile_group"].to_numpy(),
), "Profile labels differ from saved validation labels."

print("Validation profile feature reconciliation: PASS")
print("Validation cluster prediction reconciliation: PASS")
print("\nPredicted profile groups:")
print(pipeline_groups["profile_group"].value_counts().to_string())
print("\nThe saved model was applied without retraining.")

Validation profile feature reconciliation: PASS
Validation cluster prediction reconciliation: PASS

Predicted profile groups:
profile_group
Other consumption profiles    242
Night-concentrated profile      6

The saved model was applied without retraining.


In [5]:
monitoring_code = '''

def build_monitoring_results(
    bills,
    profile_groups,
    design_groups_file,
    review_increase_pct,
):
    """Create review flags without changing tariffs or charges."""
    threshold = float(review_increase_pct)
    if not np.isfinite(threshold) or threshold < 0:
        raise ValueError("Review threshold must be finite and non-negative.")

    design_groups = pd.read_csv(
        design_groups_file,
        index_col="household_id",
    )

    for frame in (bills, profile_groups, design_groups):
        if frame.index.has_duplicates:
            raise ValueError("Duplicate household keys found.")

    if not (
        set(bills.index)
        == set(profile_groups.index)
        == set(design_groups.index)
    ):
        raise ValueError("Monitoring household keys do not match.")

    results = bills.join(
        profile_groups[["profile_cluster", "profile_group"]],
        validate="one_to_one",
    ).join(
        design_groups[["design_cluster", "profile_group"]].rename(
            columns={"profile_group": "design_profile_group"}
        ),
        validate="one_to_one",
    )

    required_values = [
        "assigned_charge_gbp",
        "baseline_charge_gbp",
        "saving_gbp",
        "profile_cluster",
        "design_cluster",
    ]
    if results[required_values].isna().any().any():
        raise ValueError("Required monitoring values are missing.")

    # Ignore floating-point differences smaller than one millionth GBP.
    money_tolerance = 1e-6

    results["paying_more"] = (
        results["assigned_charge_gbp"]
        - results["baseline_charge_gbp"]
    ) > money_tolerance

    results["above_review_threshold"] = (
        results["assigned_charge_gbp"]
        - results["baseline_charge_gbp"] * (1 + threshold / 100)
    ) > money_tolerance

    results["profile_changed"] = (
        results["profile_cluster"] != results["design_cluster"]
    )
    results["review_required"] = results[
        ["paying_more", "above_review_threshold", "profile_changed"]
    ].any(axis=1)

    results["review_reasons"] = [
        "; ".join(
            reason
            for condition, reason in [
                (paying_more, "Higher charge than flat baseline"),
                (above_threshold, "Increase exceeds review threshold"),
                (profile_changed, "Profile differs from design period"),
            ]
            if condition
        )
        for paying_more, above_threshold, profile_changed
        in zip(
            results["paying_more"],
            results["above_review_threshold"],
            results["profile_changed"],
        )
    ]

    return results
'''

existing_code = module_file.read_text(encoding="utf-8")
function_marker = "\ndef build_monitoring_results("
base_code = existing_code.split(function_marker, 1)[0]

module_file.write_text(
    base_code.rstrip() + "\n" + monitoring_code,
    encoding="utf-8",
)

module_spec = importlib.util.spec_from_file_location(
    "consumption_pipeline",
    module_file,
)
pipeline = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(pipeline)

pipeline_monitoring = pipeline.build_monitoring_results(
    bills=pipeline_bills,
    profile_groups=pipeline_groups,
    design_groups_file=(
        processed_folder / "customer_design_cluster_assignments_2013.csv"
    ),
    review_increase_pct=tariff_config[
        "validation_bill_increase_limit_pct"
    ],
)

print("Households monitored:", len(pipeline_monitoring))
for column in [
    "paying_more",
    "above_review_threshold",
    "profile_changed",
    "review_required",
]:
    print(f"{column}: {int(pipeline_monitoring[column].sum())}")

display(
    pipeline_monitoring.loc[
        pipeline_monitoring["review_required"],
        [
            "assigned_tariff",
            "bill_increase_pct",
            "design_profile_group",
            "profile_group",
            "review_reasons",
        ],
    ].round(4)
)

print("Flags concern the evaluated half-year, before rebates.")
print("The 2% threshold is not a monthly bill guarantee.")
print("Review flags do not automatically change customer assignments.")

Households monitored: 248
paying_more: 7
above_review_threshold: 1
profile_changed: 2
review_required: 9


,assigned_tariff,bill_increase_pct,design_profile_group,profile_group,review_reasons
household_id,,,,,
MAC000390,Time-of-use,0.1794,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC002170,Time-of-use,0.9522,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC002517,Time-of-use,0.2272,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC002806,Time-of-use,1.8914,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC004515,Time-of-use,-17.5233,Night-concentrated profile,Other consumption profiles,Profile differs from design period
MAC004532,Time-of-use,0.1025,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC004646,Time-of-use,2.7651,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline; Increase exc...
MAC004908,Time-of-use,1.1744,Other consumption profiles,Other consumption profiles,Higher charge than flat baseline
MAC005309,Time-of-use,-10.8721,Night-concentrated profile,Other consumption profiles,Profile differs from design period


Flags concern the evaluated half-year, before rebates.
The 2% threshold is not a monthly bill guarantee.
Review flags do not automatically change customer assignments.


In [6]:
runner_code = '''
import argparse
import json
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from consumption_pipeline import (
    load_period_consumption,
    calculate_period_bills,
    predict_period_profiles,
    build_monitoring_results,
)


def main():
    parser = argparse.ArgumentParser(
        description="Evaluate saved tariffs and profile models."
    )
    parser.add_argument("--start", required=True)
    parser.add_argument("--end", required=True)
    parser.add_argument(
        "--project",
        type=Path,
        default=Path(__file__).resolve().parents[1],
    )
    args = parser.parse_args()

    root = args.project.resolve()
    processed = root / "data" / "processed"
    models = root / "models"

    config_file = processed / "locked_optional_switch_tariff_2013.json"
    assignments_file = (
        processed / "locked_customer_tariff_assignments_2013.csv"
    )
    config = json.loads(config_file.read_text(encoding="utf-8"))

    consumption = load_period_consumption(
        processed / "electricity_analysis.duckdb",
        assignments_file,
        args.start,
        args.end,
    )
    bands, bills = calculate_period_bills(
        consumption, assignments_file, config
    )
    features, groups = predict_period_profiles(
        consumption,
        models / "design_profile_kmeans_2013.joblib",
        models / "design_profile_kmeans_2013.json",
    )
    monitoring = build_monitoring_results(
        bills,
        groups,
        processed / "customer_design_cluster_assignments_2013.csv",
        config["validation_bill_increase_limit_pct"],
    )

    revenue = float(bills["assigned_charge_gbp"].sum())
    cost = float(bills["variable_cost_gbp"].sum())
    baseline = float(bills["baseline_charge_gbp"].sum())

    if revenue <= 0 or baseline <= 0:
        raise ValueError("Portfolio revenue and baseline must be positive.")

    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "_" + uuid4().hex[:8]
    )
    output = root / "reports" / "pipeline_runs" / run_id
    output.mkdir(parents=True, exist_ok=False)

    summary = {
        "run_id": run_id,
        "start_utc": consumption["interval_start_utc"].min().isoformat(),
        "end_exclusive_utc": (
            consumption["interval_start_utc"].max()
            + __import__("datetime").timedelta(minutes=30)
        ).isoformat(),
        "households": int(len(bills)),
        "intervals": int(len(consumption)),
        "energy_kwh": float(bills["energy_kwh"].sum()),
        "customer_saving_gbp": float(bills["saving_gbp"].sum()),
        "aggregate_saving_pct": float(
            100 * bills["saving_gbp"].sum() / baseline
        ),
        "contribution_margin_pct": 100 * (revenue - cost) / revenue,
        "minimum_margin_pct": config["minimum_portfolio_margin_pct"],
        "margin_constraint_pass": bool(
            100 * (revenue - cost) / revenue
            >= config["minimum_portfolio_margin_pct"]
        ),
        "review_increase_threshold_pct": config[
            "validation_bill_increase_limit_pct"
        ],
        "paying_more": int(monitoring["paying_more"].sum()),
        "above_review_threshold": int(
            monitoring["above_review_threshold"].sum()
        ),
        "profile_changed": int(monitoring["profile_changed"].sum()),
        "review_required": int(monitoring["review_required"].sum()),
        "rebates_applied": False,
        "model_retrained": False,
        "scope": "Historical evaluation from prepared database records",
        "limitations": [
            "Review thresholds apply to the evaluated period.",
            "The 2% threshold is not a monthly bill guarantee.",
            "Profile changes do not establish tariff unsuitability.",
            "Variable costs are illustrative; contribution is not net profit.",
        ],
    }

    bands.to_csv(output / "band_totals.csv")
    bills.to_csv(output / "customer_bills.csv")
    features.to_csv(output / "normalised_profiles.csv")
    monitoring.to_csv(output / "customer_monitoring.csv")
    monitoring.loc[monitoring["review_required"]].to_csv(
        output / "review_queue.csv"
    )
    for filename, content in [
        ("run_summary.json", summary),
        ("tariff_config_snapshot.json", config),
    ]:
        (output / filename).write_text(
            json.dumps(content, indent=2, allow_nan=False),
            encoding="utf-8",
        )

    print("Pipeline completed.")
    print(f"Households: {summary['households']}")
    print(f"Review required: {summary['review_required']}")
    print(f"Results saved: {output}")


if __name__ == "__main__":
    main()
'''

# Use a normal import for the period-end calculation.
runner_code = runner_code.replace(
    "from datetime import datetime, timezone",
    "from datetime import datetime, timezone, timedelta",
).replace(
    '__import__("datetime").timedelta(minutes=30)',
    "timedelta(minutes=30)",
)

runner_file = src_folder / "run_pipeline.py"
runner_file.write_text(
    runner_code.lstrip(),
    encoding="utf-8",
)

# Check syntax without running the pipeline yet.
compile(
    runner_file.read_text(encoding="utf-8"),
    str(runner_file),
    "exec",
)

print(f"Runner saved: {runner_file}")
print("Runner syntax check: PASS")
print("The standalone pipeline has not yet been executed.")

Runner saved: D:\electricity-tariff-optimisation\src\run_pipeline.py
Runner syntax check: PASS
The standalone pipeline has not yet been executed.


In [7]:
import subprocess
import sys
from time import perf_counter

command = [
    sys.executable,
    str(src_folder / "run_pipeline.py"),
    "--project",
    str(project_folder),
    "--start",
    "2013-07-01",
    "--end",
    "2014-01-01",
]

started = perf_counter()

completed = subprocess.run(
    command,
    cwd=str(project_folder),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=180,
)

elapsed_seconds = perf_counter() - started

if completed.stdout:
    print(completed.stdout)

if completed.stderr:
    print("Process diagnostics:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        f"Pipeline failed with exit code {completed.returncode}."
    )

print("Independent Python process: PASS")
print(f"Elapsed time (seconds): {elapsed_seconds:.2f}")

Pipeline completed.
Households: 248
Review required: 9
Results saved: D:\electricity-tariff-optimisation\reports\pipeline_runs\20261029T033359Z_65507703

Independent Python process: PASS
Elapsed time (seconds): 8.68


In [8]:
# Identify the output folder from this specific process.
output_lines = [
    line for line in completed.stdout.splitlines()
    if line.startswith("Results saved: ")
]
assert len(output_lines) == 1, "Could not identify the run output."

run_folder = Path(
    output_lines[0].removeprefix("Results saved: ").strip()
)

expected_files = [
    "band_totals.csv",
    "customer_bills.csv",
    "normalised_profiles.csv",
    "customer_monitoring.csv",
    "review_queue.csv",
    "run_summary.json",
    "tariff_config_snapshot.json",
]

for filename in expected_files:
    assert (run_folder / filename).is_file(), f"Missing: {filename}"
    print(f"PASS: {filename}")

exported_bills = pd.read_csv(
    run_folder / "customer_bills.csv",
    index_col="household_id",
)
exported_monitoring = pd.read_csv(
    run_folder / "customer_monitoring.csv",
    index_col="household_id",
)
exported_queue = pd.read_csv(
    run_folder / "review_queue.csv",
    index_col="household_id",
)
run_summary = json.loads(
    (run_folder / "run_summary.json").read_text(encoding="utf-8")
)
config_snapshot = json.loads(
    (run_folder / "tariff_config_snapshot.json").read_text(
        encoding="utf-8"
    )
)

assert config_snapshot == tariff_config, "Tariff snapshot differs."

pd.testing.assert_frame_equal(
    exported_bills.sort_index(),
    pipeline_bills.sort_index(),
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-6,
)
pd.testing.assert_frame_equal(
    exported_monitoring.sort_index(),
    pipeline_monitoring.sort_index().replace({"review_reasons": {"": np.nan}}),
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-6,
)

expected_queue = pipeline_monitoring.loc[
    pipeline_monitoring["review_required"]
]
assert not exported_queue.index.has_duplicates
assert set(exported_queue.index) == set(expected_queue.index)
assert len(exported_queue) == run_summary["review_required"] == 9

assert run_summary["households"] == len(exported_bills) == 248
assert np.isclose(
    run_summary["customer_saving_gbp"],
    exported_bills["saving_gbp"].sum(),
    rtol=0,
    atol=1e-6,
)
assert run_summary["rebates_applied"] is False
assert run_summary["model_retrained"] is False

print("\nExported bills and monitoring reconciliation: PASS")
print("Review queue reconciliation: PASS")
print("Run summary reconciliation: PASS")
print(f"Verified run: {run_folder.name}")

PASS: band_totals.csv
PASS: customer_bills.csv
PASS: normalised_profiles.csv
PASS: customer_monitoring.csv
PASS: review_queue.csv
PASS: run_summary.json
PASS: tariff_config_snapshot.json

Exported bills and monitoring reconciliation: PASS
Review queue reconciliation: PASS
Run summary reconciliation: PASS
Verified run: 20261029T033359Z_65507703


In [9]:
import tempfile
import duckdb

test_results = []


def expect_value_error(case, action, expected_message):
    try:
        action()
    except ValueError as error:
        assert expected_message in str(error), (
            f"{case}: unexpected error: {error}"
        )
        test_results.append({"case": case, "passed": True})
    else:
        raise AssertionError(f"{case}: invalid input was accepted.")


with tempfile.TemporaryDirectory() as temporary_folder:
    temporary_folder = Path(temporary_folder)
    test_database = temporary_folder / "test.duckdb"
    test_assignments = temporary_folder / "assignments.csv"

    pd.DataFrame({
        "household_id": ["TEST001"],
        "assigned_tariff": ["Time-of-use"],
    }).to_csv(test_assignments, index=False)

    valid_data = pd.DataFrame({
        "household_id": ["TEST001", "TEST001"],
        "interval_start_utc": pd.to_datetime([
            "2013-07-01 00:00:00",
            "2013-07-01 00:30:00",
        ]),
        "energy_kwh": [1.0, 2.0],
    })

    def write_test_data(frame):
        with duckdb.connect(str(test_database)) as connection:
            connection.register("test_input", frame)
            connection.execute("""
                CREATE OR REPLACE TABLE analysis_consumption_2013 AS
                SELECT * FROM test_input
            """)

    def load_test_period(start="2013-07-01 00:00:00",
                         end="2013-07-01 01:00:00"):
        return pipeline.load_period_consumption(
            test_database,
            test_assignments,
            start,
            end,
        )

    write_test_data(valid_data)
    loaded = load_test_period()
    assert len(loaded) == 2
    assert loaded["energy_kwh"].sum() == 3.0
    test_results.append({"case": "Valid input accepted", "passed": True})

    expect_value_error(
        "Reversed period",
        lambda: load_test_period(
            "2013-07-02", "2013-07-01"
        ),
        "start before end",
    )

    expect_value_error(
        "Misaligned period boundary",
        lambda: load_test_period(
            "2013-07-01 00:15:00"
        ),
        "align to half-hours",
    )

    write_test_data(valid_data.iloc[:1])
    expect_value_error(
        "Missing interval",
        load_test_period,
        "Incomplete household interval coverage",
    )

    write_test_data(pd.concat(
        [valid_data, valid_data.iloc[:1]],
        ignore_index=True,
    ))
    expect_value_error(
        "Duplicate interval",
        load_test_period,
        "Duplicate household intervals",
    )

    negative_data = valid_data.copy()
    negative_data.loc[0, "energy_kwh"] = -1.0
    write_test_data(negative_data)
    expect_value_error(
        "Negative energy",
        load_test_period,
        "finite and non-negative",
    )

    missing_data = valid_data.copy()
    missing_data.loc[0, "energy_kwh"] = np.nan
    write_test_data(missing_data)
    expect_value_error(
        "Missing energy",
        load_test_period,
        "finite and non-negative",
    )

    wrong_household = valid_data.copy()
    wrong_household["household_id"] = "UNEXPECTED"
    write_test_data(wrong_household)
    expect_value_error(
        "Unexpected household",
        load_test_period,
        "households differ",
    )

test_results = pd.DataFrame(test_results)
display(test_results)

assert test_results["passed"].all()
print("Input validation tests: PASS")
print("Original database and saved results were not modified.")

,case,passed
0,Valid input accepted,True
1,Reversed period,True
2,Misaligned period boundary,True
3,Missing interval,True
4,Duplicate interval,True
5,Negative energy,True
6,Missing energy,True
7,Unexpected household,True


Input validation tests: PASS
Original database and saved results were not modified.


In [10]:
with tempfile.TemporaryDirectory() as temporary_folder:
    temporary_folder = Path(temporary_folder)
    assignments_path = temporary_folder / "assignments.csv"

    pd.DataFrame({
        "household_id": ["TEST_TOU", "TEST_FLAT"],
        "assigned_tariff": ["Time-of-use", "Flat"],
    }).to_csv(assignments_path, index=False)

    # Winter Monday: off-peak, shoulder, peak.
    timestamps = pd.to_datetime([
        "2013-01-07 00:00:00Z",
        "2013-01-07 07:00:00Z",
        "2013-01-07 16:00:00Z",
    ])

    sample_consumption = pd.DataFrame({
        "household_id": ["TEST_TOU"] * 3 + ["TEST_FLAT"] * 3,
        "interval_start_utc": list(timestamps) * 2,
        "energy_kwh": [1.0, 2.0, 3.0] * 2,
    })

    _, sample_bills = pipeline.calculate_period_bills(
        sample_consumption,
        assignments_path,
        tariff_config,
    )

    # Manual calculations:
    # Baseline: 6 * 0.14228 = 0.85368 GBP
    # TOU: 1 * 0.08 + 2 * 0.13 + 3 * 0.22 = 1.00 GBP
    # Variable cost: 1 * 0.06 + 2 * 0.10 + 3 * 0.15 = 0.71 GBP
    assert np.isclose(
        sample_bills.loc["TEST_TOU", "baseline_charge_gbp"],
        0.85368, rtol=0, atol=1e-10,
    )
    assert np.isclose(
        sample_bills.loc["TEST_TOU", "assigned_charge_gbp"],
        1.00, rtol=0, atol=1e-10,
    )
    assert np.isclose(
        sample_bills.loc["TEST_TOU", "saving_gbp"],
        -0.14632, rtol=0, atol=1e-10,
    )
    assert np.isclose(
        sample_bills.loc["TEST_TOU", "variable_cost_gbp"],
        0.71, rtol=0, atol=1e-10,
    )
    assert np.isclose(
        sample_bills.loc["TEST_FLAT", "assigned_charge_gbp"],
        0.85368, rtol=0, atol=1e-10,
    )
    assert sample_bills.loc["TEST_FLAT", "saving_gbp"] == 0

    # Test below, exactly at, and above the 2% review threshold.
    household_ids = ["BELOW", "EXACT", "ABOVE"]
    boundary_bills = pd.DataFrame({
        "baseline_charge_gbp": [100.0] * 3,
        "assigned_charge_gbp": [101.99, 102.0, 102.01],
        "saving_gbp": [-1.99, -2.0, -2.01],
    }, index=pd.Index(household_ids, name="household_id"))

    boundary_groups = pd.DataFrame({
        "profile_cluster": [0, 0, 0],
        "profile_group": ["Other consumption profiles"] * 3,
    }, index=boundary_bills.index)

    design_groups_path = temporary_folder / "design_groups.csv"
    pd.DataFrame({
        "design_cluster": [0, 0, 0],
        "profile_group": ["Other consumption profiles"] * 3,
    }, index=boundary_bills.index).to_csv(design_groups_path)

    boundary_results = pipeline.build_monitoring_results(
        boundary_bills,
        boundary_groups,
        design_groups_path,
        review_increase_pct=2.0,
    )

    assert boundary_results["above_review_threshold"].tolist() == [
        False, False, True
    ]
    assert boundary_results["paying_more"].all()
    assert not boundary_results["profile_changed"].any()

    print("Manual TOU bill calculation: PASS")
    print("Flat tariff assignment retained: PASS")
    print("Variable-cost calculation: PASS")
    print("Below / exact / above 2% threshold: PASS")

    display(boundary_results[
        [
            "baseline_charge_gbp",
            "assigned_charge_gbp",
            "above_review_threshold",
        ]
    ])

print("Original project inputs were not modified.")

Manual TOU bill calculation: PASS
Flat tariff assignment retained: PASS
Variable-cost calculation: PASS
Below / exact / above 2% threshold: PASS


,baseline_charge_gbp,assigned_charge_gbp,above_review_threshold
household_id,,,
BELOW,100.0,101.99,False
EXACT,100.0,102.00,False
ABOVE,100.0,102.01,True


Original project inputs were not modified.


In [11]:
from importlib.metadata import version
from datetime import datetime, timezone

validation_folder = project_folder / "reports" / "validation"
validation_folder.mkdir(parents=True, exist_ok=True)

assert test_results["passed"].all()

billing_checks = pd.DataFrame([
    {"case": "Manual TOU bill calculation", "passed": True},
    {"case": "Flat tariff assignment retained", "passed": True},
    {"case": "Variable-cost calculation", "passed": True},
    {"case": "Below / exact / above 2% threshold", "passed": True},
])

all_checks = pd.concat(
    [test_results, billing_checks],
    ignore_index=True,
)
all_checks.to_csv(
    validation_folder / "pipeline_function_checks.csv",
    index=False,
)

validation_record = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "verified_run_id": run_summary["run_id"],
    "function_checks_passed": int(all_checks["passed"].sum()),
    "independent_process_exit_code": int(completed.returncode),
    "observed_runtime_seconds": round(elapsed_seconds, 2),
    "scope": [
        "Prepared-data input validation",
        "Manual sample billing and threshold checks",
        "Saved H2 billing and profile reconciliation",
        "Independent process execution and exported-result checks",
    ],
    "limitations": [
        "Raw CSV ingestion is outside this pipeline.",
        "Automated scheduling has not been configured.",
        "Runtime is not a measurement of manual time saved.",
        "Checks do not validate supplier cost assumptions.",
    ],
}

summary_file = (
    validation_folder / "pipeline_validation_summary.json"
)
summary_file.write_text(
    json.dumps(validation_record, indent=2, allow_nan=False),
    encoding="utf-8",
)

requirements_file = project_folder / "requirements.txt"
existing_lines = requirements_file.read_text(
    encoding="utf-8-sig"
).splitlines()

updated_lines = [
    line for line in existing_lines
    if not line.strip().lower().startswith(
        ("scikit-learn==", "joblib==")
    )
]
updated_lines += [
    f"scikit-learn=={version('scikit-learn')}",
    f"joblib=={version('joblib')}",
]

requirements_file.write_text(
    "\n".join(updated_lines).rstrip() + "\n",
    encoding="utf-8",
)

guide_lines = [
    "# Consumption analysis pipeline",
    "",
    "Run from the project root using the project's Python environment:",
    "",
    "```bash",
    "python src/run_pipeline.py --start 2013-07-01 --end 2014-01-01",
    "```",
    "",
    "## Inputs and period conventions",
    "",
    "- Reuses the prepared DuckDB database; raw CSV ingestion is excluded.",
    "- Dates use UTC interval starts: start inclusive, end exclusive.",
    "- Period boundaries must align to half-hours.",
    "- All assigned households require complete interval coverage.",
    "- Profiles require all 96 features and positive mean consumption.",
    "- Uses saved tariff prices, assignments and the profile model.",
    "- Load joblib models only from trusted sources.",
    "",
    "## Outputs",
    "",
    "Each successful run creates a folder under reports/pipeline_runs/.",
    "Outputs include band totals, bills, normalised profiles, monitoring,",
    "a review queue, a run summary and a tariff configuration snapshot.",
    "",
    "## Interpretation",
    "",
    "- Charges are before rebates.",
    "- The 2% threshold applies to the evaluated period.",
    "- It is not a monthly bill guarantee.",
    "- Profile changes do not establish tariff unsuitability.",
    "- Review flags require human assessment.",
    "- Variable costs are illustrative; contribution is not net profit.",
    "- No training, optimisation or automatic switching occurs.",
    "- Scheduling has not been configured.",
    "",
    "## Checks",
    "",
    "See notebooks/08_pipeline_development.ipynb for checks.",
    "Check records are saved under reports/validation/.",
]

guide_file = project_folder / "reports" / "pipeline_usage.md"
guide_file.write_text(
    "\n".join(guide_lines) + "\n",
    encoding="utf-8",
)

print(f"Function checks recorded: {len(all_checks)}")
print("Pipeline validation summary saved.")
print("Pipeline usage guide saved.")
print("scikit-learn and joblib versions added to requirements.txt.")
print("Scheduling has not been configured.")

Function checks recorded: 12
Pipeline validation summary saved.
Pipeline usage guide saved.
scikit-learn and joblib versions added to requirements.txt.
Scheduling has not been configured.


In [13]:
import importlib.util
import pandas as pd
import numpy as np

module_file = project_folder / "src" / "consumption_pipeline.py"
spec = importlib.util.spec_from_file_location(
    "consumption_pipeline", module_file
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

processed_folder = project_folder / "data" / "processed"

july_consumption = pipeline.load_period_consumption(
    database_file=processed_folder / "electricity_analysis.duckdb",
    assignments_file=(
        processed_folder / "locked_customer_tariff_assignments_2013.csv"
    ),
    start_utc="2013-07-01",
    end_utc="2013-08-01",
)

july_energy_review = (
    july_consumption.groupby("household_id")
    .agg(
        intervals=("energy_kwh", "size"),
        total_energy_kwh=("energy_kwh", "sum"),
        zero_intervals=("energy_kwh", lambda values: values.eq(0).sum()),
    )
)

july_energy_review["zero_interval_pct"] = (
    100 * july_energy_review["zero_intervals"]
    / july_energy_review["intervals"]
)

zero_energy_households = july_energy_review.loc[
    july_energy_review["total_energy_kwh"].eq(0)
]

print("July households:", len(july_energy_review))
print("Households with zero recorded July consumption:",
      len(zero_energy_households))
display(zero_energy_households)

print("Diagnostic only; no readings or model files were changed.")
print("Zero readings alone do not establish their cause.")

July households: 248
Households with zero recorded July consumption: 1


,intervals,total_energy_kwh,zero_intervals,zero_interval_pct
household_id,,,,
MAC005263,1488,0.0,1488,100.0


Diagnostic only; no readings or model files were changed.
Zero readings alone do not establish their cause.


In [14]:
import ast
import shutil

replacement_function = '''
def predict_period_profiles(consumption, model_file, metadata_file):
    """Predict eligible profiles; retain zero-consumption households."""
    import json
    import joblib

    metadata = json.loads(
        Path(metadata_file).read_text(encoding="utf-8")
    )
    local_start = consumption["interval_start_utc"].dt.tz_convert(
        "Europe/London"
    )
    profile_data = consumption.assign(
        day_type=np.where(
            local_start.dt.dayofweek < 5, "Weekday", "Weekend"
        ),
        slot=local_start.dt.hour * 2 + local_start.dt.minute // 30,
    )
    expected_columns = pd.MultiIndex.from_product(
        [["Weekday", "Weekend"], range(48)],
        names=["day_type", "slot"],
    )
    profiles = (
        profile_data
        .groupby(["household_id", "day_type", "slot"])["energy_kwh"]
        .mean()
        .unstack(["day_type", "slot"])
        .reindex(columns=expected_columns)
        .sort_index()
    )

    values = profiles.to_numpy(dtype=float)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Incomplete or invalid consumption profiles.")

    profile_means = profiles.mean(axis=1)
    eligible = profile_means.gt(0)

    # Zero-consumption households retain missing normalised features.
    normalised = profiles.div(
        profile_means.where(eligible), axis=0
    )
    normalised.columns = [
        f"{day_type.lower()}_{slot:02d}"
        for day_type, slot in expected_columns
    ]
    if list(normalised.columns) != metadata["feature_columns"]:
        raise ValueError("Features differ from saved model metadata.")

    # Load only trusted model files created in this project.
    model = joblib.load(model_file)
    if model.n_features_in_ != normalised.shape[1]:
        raise ValueError("Model feature count does not match profiles.")

    results = pd.DataFrame(index=profiles.index)
    results["profile_cluster"] = pd.Series(
        pd.NA, index=profiles.index, dtype="Int64"
    )
    results["profile_group"] = "Unavailable: zero consumption"
    results["profile_status"] = "zero_consumption"

    labels = {
        int(key): value
        for key, value in metadata["profile_labels"].items()
    }

    if eligible.any():
        predictions = model.predict(
            normalised.loc[eligible].to_numpy()
        )
        if not set(predictions).issubset(labels):
            raise ValueError("Predicted cluster has no saved label.")

        results.loc[eligible, "profile_cluster"] = predictions
        results.loc[eligible, "profile_group"] = [
            labels[int(cluster)] for cluster in predictions
        ]
        results.loc[eligible, "profile_status"] = "available"

    return normalised, results
'''

# Replace only this function, preserving all other module functions.
source = module_file.read_text(encoding="utf-8")
tree = ast.parse(source)
matches = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "predict_period_profiles"
]
assert len(matches) == 1

node = matches[0]
lines = source.splitlines(keepends=True)
updated_source = (
    "".join(lines[:node.lineno - 1])
    + replacement_function.strip() + "\n"
    + "".join(lines[node.end_lineno:])
)
compile(updated_source, str(module_file), "exec")

backup_file = module_file.with_suffix(".py.before_monthly_profiles.bak")
if not backup_file.exists():
    shutil.copy2(module_file, backup_file)

module_file.write_text(updated_source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "consumption_pipeline", module_file
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

july_profiles, july_groups = pipeline.predict_period_profiles(
    july_consumption,
    project_folder / "models" / "design_profile_kmeans_2013.joblib",
    project_folder / "models" / "design_profile_kmeans_2013.json",
)

assert len(july_groups) == 248
assert july_groups.loc["MAC005263", "profile_status"] == "zero_consumption"
assert pd.isna(july_groups.loc["MAC005263", "profile_cluster"])
assert july_profiles.loc["MAC005263"].isna().all()

print("Households retained:", len(july_groups))
print("\nProfile status:")
print(july_groups["profile_status"].value_counts().to_string())
print("\nZero-consumption profile handling: PASS")
print("Monitoring function still needs updating before a full run.")

Households retained: 248

Profile status:
profile_status
available           247
zero_consumption      1

Zero-consumption profile handling: PASS
Monitoring function still needs updating before a full run.


In [15]:
monitoring_replacement = '''
def build_monitoring_results(
    bills,
    profile_groups,
    design_groups_file,
    review_increase_pct,
):
    """Flag charge issues, profile changes and unavailable profiles."""
    threshold = float(review_increase_pct)
    if not np.isfinite(threshold) or threshold < 0:
        raise ValueError("Review threshold must be finite and non-negative.")

    design_groups = pd.read_csv(
        design_groups_file, index_col="household_id"
    )
    for frame in (bills, profile_groups, design_groups):
        if frame.index.has_duplicates:
            raise ValueError("Duplicate household keys found.")

    if not (
        set(bills.index)
        == set(profile_groups.index)
        == set(design_groups.index)
    ):
        raise ValueError("Monitoring household keys do not match.")

    groups = profile_groups.copy()
    if "profile_status" not in groups:
        groups["profile_status"] = "available"

    results = bills.join(
        groups[["profile_cluster", "profile_group", "profile_status"]],
        validate="one_to_one",
    ).join(
        design_groups[["design_cluster", "profile_group"]].rename(
            columns={"profile_group": "design_profile_group"}
        ),
        validate="one_to_one",
    )

    money_columns = [
        "assigned_charge_gbp", "baseline_charge_gbp", "saving_gbp"
    ]
    if not np.isfinite(
        results[money_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Invalid monitoring charge values.")

    if results["design_cluster"].isna().any():
        raise ValueError("Missing design cluster.")

    if not results["profile_status"].isin(
        ["available", "zero_consumption"]
    ).all():
        raise ValueError("Unknown profile status.")

    available = results["profile_status"].eq("available")
    if results.loc[available, "profile_cluster"].isna().any():
        raise ValueError("Available profiles require cluster predictions.")
    if results.loc[~available, "profile_cluster"].notna().any():
        raise ValueError("Unavailable profiles must not have predictions.")

    tolerance = 1e-6
    results["paying_more"] = (
        results["assigned_charge_gbp"]
        - results["baseline_charge_gbp"]
    ) > tolerance
    results["above_review_threshold"] = (
        results["assigned_charge_gbp"]
        - results["baseline_charge_gbp"] * (1 + threshold / 100)
    ) > tolerance

    results["profile_unavailable"] = ~available
    results["profile_changed"] = pd.Series(
        pd.NA, index=results.index, dtype="boolean"
    )
    results.loc[available, "profile_changed"] = (
        results.loc[available, "profile_cluster"]
        != results.loc[available, "design_cluster"]
    ).astype(bool)

    results["review_required"] = (
        results["paying_more"]
        | results["above_review_threshold"]
        | results["profile_changed"].fillna(False)
        | results["profile_unavailable"]
    )

    reasons = []
    for _, row in results.iterrows():
        items = []
        if row["paying_more"]:
            items.append("Higher charge than flat baseline")
        if row["above_review_threshold"]:
            items.append("Increase exceeds review threshold")
        if row["profile_unavailable"]:
            items.append("Profile unavailable: zero recorded consumption")
        elif row["profile_changed"]:
            items.append("Profile differs from design period")
        reasons.append("; ".join(items))

    results["review_reasons"] = reasons
    return results
'''

source = module_file.read_text(encoding="utf-8")
tree = ast.parse(source)
matches = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "build_monitoring_results"
]
assert len(matches) == 1

node = matches[0]
lines = source.splitlines(keepends=True)
updated_source = (
    "".join(lines[:node.lineno - 1])
    + monitoring_replacement.strip() + "\n"
    + "".join(lines[node.end_lineno:])
)
compile(updated_source, str(module_file), "exec")

backup_file = module_file.with_suffix(".py.before_monthly_monitoring.bak")
if not backup_file.exists():
    shutil.copy2(module_file, backup_file)

module_file.write_text(updated_source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "consumption_pipeline", module_file
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

config = json.loads(
    (
        processed_folder / "locked_optional_switch_tariff_2013.json"
    ).read_text(encoding="utf-8")
)

july_bands, july_bills = pipeline.calculate_period_bills(
    july_consumption,
    processed_folder / "locked_customer_tariff_assignments_2013.csv",
    config,
)

july_monitoring = pipeline.build_monitoring_results(
    july_bills,
    july_groups,
    processed_folder / "customer_design_cluster_assignments_2013.csv",
    review_increase_pct=2.0,
)

zero_case = july_monitoring.loc["MAC005263"]

assert len(july_monitoring) == 248
assert zero_case["baseline_charge_gbp"] == 0
assert zero_case["assigned_charge_gbp"] == 0
assert pd.isna(zero_case["bill_increase_pct"])
assert pd.isna(zero_case["profile_changed"])
assert zero_case["profile_unavailable"]
assert zero_case["review_required"]
assert not zero_case["paying_more"]
assert not zero_case["above_review_threshold"]

print("Zero-consumption monitoring checks: PASS")
print("Households monitored:", len(july_monitoring))
print("\nZero-consumption case:")
display(july_monitoring.loc[
    ["MAC005263"],
    [
        "assigned_charge_gbp",
        "bill_increase_pct",
        "profile_status",
        "profile_changed",
        "review_required",
        "review_reasons",
    ],
])

Zero-consumption monitoring checks: PASS
Households monitored: 248

Zero-consumption case:


,assigned_charge_gbp,bill_increase_pct,profile_status,profile_changed,review_required,review_reasons
household_id,,,,,,
MAC005263,0.0,NaN,zero_consumption,<NA>,True,Profile unavailable: zero recorded consumption


In [16]:
# Add monthly monitoring fields to the standalone runner.
runner_file = project_folder / "src" / "run_pipeline.py"
runner_source = runner_file.read_text(encoding="utf-8")

marker = '        "profile_unavailable":'
anchor = '        "profile_changed": int(monitoring["profile_changed"].sum()),'

if marker not in runner_source:
    assert runner_source.count(anchor) == 1, \
        "Could not locate the runner summary section."

    additional_fields = "\n".join([
        '        "profile_unavailable": int(',
        '            monitoring["profile_unavailable"].sum()',
        '        ),',
        '        "profile_comparable_households": int(',
        '            monitoring["profile_changed"].notna().sum()',
        '        ),',
        '        "zero_baseline_households": int(',
        '            bills["baseline_charge_gbp"].eq(0).sum()',
        '        ),',
    ])

    runner_source = runner_source.replace(
        anchor,
        anchor + "\n" + additional_fields,
    )

    compile(runner_source, str(runner_file), "exec")

    runner_backup = runner_file.with_suffix(".py.before_monthly.bak")
    if not runner_backup.exists():
        shutil.copy2(runner_file, runner_backup)

    runner_file.write_text(runner_source, encoding="utf-8")

# Run July in an independent process.
july_process = subprocess.run(
    [
        sys.executable,
        str(runner_file),
        "--project", str(project_folder),
        "--start", "2013-07-01",
        "--end", "2013-08-01",
    ],
    cwd=str(project_folder),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    timeout=180,
)

print(july_process.stdout)

if july_process.stderr:
    print("Process diagnostics:")
    print(july_process.stderr)

if july_process.returncode != 0:
    raise RuntimeError("July standalone pipeline failed.")

output_lines = [
    line for line in july_process.stdout.splitlines()
    if line.startswith("Results saved: ")
]
assert len(output_lines) == 1

july_run_folder = Path(
    output_lines[0].removeprefix("Results saved: ").strip()
)
july_summary = json.loads(
    (july_run_folder / "run_summary.json").read_text(encoding="utf-8")
)

exported_july_monitoring = pd.read_csv(
    july_run_folder / "customer_monitoring.csv",
    index_col="household_id",
)

assert july_summary["households"] == 248
assert july_summary["intervals"] == 369024
assert july_summary["profile_unavailable"] == 1
assert july_summary["profile_comparable_households"] == 247
assert july_summary["zero_baseline_households"] == 1

assert pd.isna(
    exported_july_monitoring.loc["MAC005263", "profile_changed"]
)
assert exported_july_monitoring.loc[
    "MAC005263", "profile_status"
] == "zero_consumption"

print("July standalone execution and zero-profile export: PASS")

for field in [
    "customer_saving_gbp",
    "contribution_margin_pct",
    "paying_more",
    "above_review_threshold",
    "profile_changed",
    "profile_unavailable",
    "profile_comparable_households",
    "review_required",
]:
    print(f"{field}: {july_summary[field]}")

print("The 2% monthly threshold is for review, not a bill guarantee.")

Pipeline completed.
Households: 248
Review required: 6
Results saved: D:\electricity-tariff-optimisation\reports\pipeline_runs\20261029T033747Z_c63c656a

July standalone execution and zero-profile export: PASS
customer_saving_gbp: 618.8411654668952
contribution_margin_pct: 25.52890974263111
paying_more: 4
above_review_threshold: 1
profile_changed: 1
profile_unavailable: 1
profile_comparable_households: 247
review_required: 6
The 2% monthly threshold is for review, not a bill guarantee.


In [17]:
# Retain successful monthly runs if this cell is rerun.
if "monthly_run_folders" not in globals():
    monthly_run_folders = {}

monthly_run_folders["2013-07"] = july_run_folder

month_boundaries = pd.date_range(
    "2013-08-01",
    "2014-01-01",
    freq="MS",
)

for start, end in zip(month_boundaries[:-1], month_boundaries[1:]):
    month_label = start.strftime("%Y-%m")

    if month_label in monthly_run_folders:
        print(f"{month_label}: using saved run.")
        continue

    process = subprocess.run(
        [
            sys.executable,
            str(runner_file),
            "--project", str(project_folder),
            "--start", start.strftime("%Y-%m-%d"),
            "--end", end.strftime("%Y-%m-%d"),
        ],
        cwd=str(project_folder),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=180,
    )

    if process.returncode != 0:
        print(process.stdout)
        print(process.stderr)
        raise RuntimeError(f"{month_label}: pipeline failed.")

    output_lines = [
        line for line in process.stdout.splitlines()
        if line.startswith("Results saved: ")
    ]
    assert len(output_lines) == 1

    folder = Path(
        output_lines[0].removeprefix("Results saved: ").strip()
    )
    summary = json.loads(
        (folder / "run_summary.json").read_text(encoding="utf-8")
    )

    expected_intervals = 248 * (end - start).days * 48
    assert summary["households"] == 248
    assert summary["intervals"] == expected_intervals
    assert summary["rebates_applied"] is False
    assert summary["model_retrained"] is False

    monthly_run_folders[month_label] = folder
    print(
        f"{month_label}: PASS | "
        f"saving GBP {summary['customer_saving_gbp']:.2f} | "
        f"review cases {summary['review_required']}",
        flush=True,
    )

assert len(monthly_run_folders) == 6

# Save an explicit manifest rather than selecting folders by latest date.
manifest = {
    month: {
        "run_id": folder.name,
        "folder": folder.relative_to(project_folder).as_posix(),
    }
    for month, folder in sorted(monthly_run_folders.items())
}

manifest_file = (
    project_folder / "reports" / "validation"
    / "monthly_monitoring_run_manifest_2013.json"
)
manifest_file.parent.mkdir(parents=True, exist_ok=True)
manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\nSix monthly runs available.")
print("Run manifest saved:", manifest_file)
print("Monthly figures have not yet been reconciled to H2 totals.")

2013-08: PASS | saving GBP 659.47 | review cases 4
2013-09: PASS | saving GBP 626.98 | review cases 6
2013-10: PASS | saving GBP 580.80 | review cases 26
2013-11: PASS | saving GBP 664.11 | review cases 36
2013-12: PASS | saving GBP 775.29 | review cases 35

Six monthly runs available.
Run manifest saved: D:\electricity-tariff-optimisation\reports\validation\monthly_monitoring_run_manifest_2013.json
Monthly figures have not yet been reconciled to H2 totals.


In [18]:
monthly_frames = []

for month, folder in sorted(monthly_run_folders.items()):
    frame = pd.read_csv(
        folder / "customer_monitoring.csv",
        index_col="household_id",
    )
    assert len(frame) == 248
    assert frame.index.is_unique

    frame = frame.reset_index()
    frame["month_utc"] = month
    monthly_frames.append(frame)

monthly_monitoring = pd.concat(monthly_frames, ignore_index=True)

assert len(monthly_monitoring) == 248 * 6
assert not monthly_monitoring.duplicated(
    ["household_id", "month_utc"]
).any()

saved_h2 = pd.read_csv(
    processed_folder / "validation_locked_tariff_customer_impact_2013.csv",
    index_col="household_id",
)
assert saved_h2.index.is_unique

additive_columns = [
    "energy_kwh",
    "baseline_charge_gbp",
    "tou_charge_gbp",
    "assigned_charge_gbp",
    "saving_gbp",
    "variable_cost_gbp",
]

assert np.isfinite(
    monthly_monitoring[additive_columns].to_numpy(dtype=float)
).all()

monthly_totals = monthly_monitoring.groupby(
    "household_id"
)[additive_columns].sum()

assert set(monthly_totals.index) == set(saved_h2.index)
saved_h2 = saved_h2.reindex(monthly_totals.index)

# Confirm that every month's tariff assignment remained locked.
saved_assignments = saved_h2["assigned_tariff"]
expected_assignments = monthly_monitoring["household_id"].map(
    saved_assignments
)
assert monthly_monitoring["assigned_tariff"].eq(
    expected_assignments
).all()

differences = (
    monthly_totals - saved_h2[additive_columns]
).abs()

assert np.isfinite(differences.to_numpy()).all()
assert differences.le(1e-6).all().all(), \
    "Monthly totals differ from saved H2 results."

reconciliation = pd.DataFrame({
    "field": additive_columns,
    "max_absolute_difference": differences.max().to_numpy(),
    "households_outside_tolerance": (
        differences.gt(1e-6).sum().to_numpy()
    ),
})

validation_folder = project_folder / "reports" / "validation"
reconciliation.to_csv(
    validation_folder / "monthly_to_h2_reconciliation_2013.csv",
    index=False,
)

print("Monthly customer rows:", len(monthly_monitoring))
print("Locked tariff assignments: PASS")
print("Monthly-to-H2 household reconciliation: PASS")
display(reconciliation)

print(
    "Total H2 saving before rebates (GBP): "
    f"{monthly_totals['saving_gbp'].sum():,.4f}"
)
print("Monthly percentages and profile labels were not added together.")

Monthly customer rows: 1488
Locked tariff assignments: PASS
Monthly-to-H2 household reconciliation: PASS


,field,max_absolute_difference,households_outside_tolerance
0,energy_kwh,9.094947e-13,0
1,baseline_charge_gbp,4.547474e-13,0
2,tou_charge_gbp,4.547474e-13,0
3,assigned_charge_gbp,4.547474e-13,0
4,saving_gbp,4.547474e-13,0
5,variable_cost_gbp,2.273737e-13,0


Total H2 saving before rebates (GBP): 3,925.4984
Monthly percentages and profile labels were not added together.


In [19]:
# Restore nullable boolean types after CSV loading.
flag_columns = [
    "paying_more",
    "above_review_threshold",
    "profile_changed",
    "profile_unavailable",
    "review_required",
]

for column in flag_columns:
    values = monthly_monitoring[column].astype("string").str.lower()
    assert values.dropna().isin(["true", "false"]).all(), column

    monthly_monitoring[column] = values.map({
        "true": True,
        "false": False,
    }).astype("boolean")

for column in flag_columns:
    if column != "profile_changed":
        assert monthly_monitoring[column].notna().all(), column

assert monthly_monitoring["profile_changed"].isna().equals(
    monthly_monitoring["profile_unavailable"].astype(bool)
)

monthly_monitoring["charge_review"] = (
    monthly_monitoring["paying_more"]
    | monthly_monitoring["above_review_threshold"]
)

monthly_summary = (
    monthly_monitoring.groupby("month_utc")
    .agg(
        households=("household_id", "nunique"),
        customer_saving_gbp=("saving_gbp", "sum"),
        baseline_charge_gbp=("baseline_charge_gbp", "sum"),
        assigned_charge_gbp=("assigned_charge_gbp", "sum"),
        variable_cost_gbp=("variable_cost_gbp", "sum"),
        paying_more=("paying_more", "sum"),
        above_review_threshold=("above_review_threshold", "sum"),
        profile_changed=("profile_changed", "sum"),
        profile_unavailable=("profile_unavailable", "sum"),
        review_required=("review_required", "sum"),
    )
)

monthly_summary["aggregate_saving_pct"] = (
    100 * monthly_summary["customer_saving_gbp"]
    / monthly_summary["baseline_charge_gbp"]
)
monthly_summary["contribution_margin_pct"] = (
    100
    * (
        monthly_summary["assigned_charge_gbp"]
        - monthly_summary["variable_cost_gbp"]
    )
    / monthly_summary["assigned_charge_gbp"]
)

display(monthly_summary[
    [
        "customer_saving_gbp",
        "aggregate_saving_pct",
        "contribution_margin_pct",
        "paying_more",
        "above_review_threshold",
        "profile_changed",
        "profile_unavailable",
        "review_required",
    ]
].round(4))

print("Counts for different review reasons may overlap.")
print("Profile changes are measured against the fixed H1 group.")
print("Monthly profile changes do not establish their cause.")
print("Monthly 2% flags are review thresholds, not bill guarantees.")

,customer_saving_gbp,aggregate_saving_pct,contribution_margin_pct,paying_more,above_review_threshold,profile_changed,profile_unavailable,review_required
month_utc,,,,,,,,
2013-07,618.8412,6.5176,25.5289,4,1,1,1,6
2013-08,659.4738,6.9744,25.4702,3,1,1,0,4
2013-09,626.9834,6.0605,25.3975,4,1,2,0,6
2013-10,580.8027,4.9716,25.5281,23,6,2,1,26
2013-11,664.1118,4.8827,25.4810,35,9,1,0,36
2013-12,775.2854,5.1560,25.4970,33,7,1,1,35


Counts for different review reasons may overlap.
Profile changes are measured against the fixed H1 group.
Monthly profile changes do not establish their cause.
Monthly 2% flags are review thresholds, not bill guarantees.


In [20]:
month_order = [
    "2013-07", "2013-08", "2013-09",
    "2013-10", "2013-11", "2013-12",
]


def longest_true_streak(values):
    longest = 0
    current = 0
    for value in values:
        current = current + 1 if value else 0
        longest = max(longest, current)
    return longest


# Small known examples check the consecutive-month calculation.
assert longest_true_streak([True, True, False, True]) == 2
assert longest_true_streak([False] * 6) == 0
assert longest_true_streak([True] * 6) == 6

paying_more_matrix = (
    monthly_monitoring.pivot(
        index="household_id",
        columns="month_utc",
        values="paying_more",
    )
    .reindex(columns=month_order)
)
assert paying_more_matrix.notna().all().all()

household_monthly_review = (
    monthly_monitoring.groupby("household_id")
    .agg(
        months_paying_more=("paying_more", "sum"),
        months_above_threshold=("above_review_threshold", "sum"),
        months_profile_changed=("profile_changed", "sum"),
        months_profile_unavailable=("profile_unavailable", "sum"),
        worst_month_increase_pct=("bill_increase_pct", "max"),
    )
)

household_monthly_review["longest_paying_more_streak"] = (
    paying_more_matrix.apply(
        lambda row: longest_true_streak(row.tolist()),
        axis=1,
    )
)
household_monthly_review["h2_saving_gbp"] = saved_h2["saving_gbp"]
household_monthly_review["h2_saving_pct"] = saved_h2["saving_pct"]

household_monthly_review["h2_saver_with_loss_month"] = (
    household_monthly_review["h2_saving_gbp"].gt(1e-6)
    & household_monthly_review["months_paying_more"].gt(0)
)

print(
    "Households paying more in at least one month:",
    int(household_monthly_review["months_paying_more"].gt(0).sum()),
)
print(
    "H2 savers with at least one loss month:",
    int(household_monthly_review["h2_saver_with_loss_month"].sum()),
)
print(
    "Households paying more for at least three consecutive months:",
    int(
        household_monthly_review["longest_paying_more_streak"]
        .ge(3).sum()
    ),
)

print("\nHouseholds with monthly charge increases:")
display(
    household_monthly_review.loc[
        household_monthly_review["months_paying_more"].gt(0)
    ].sort_values(
        ["longest_paying_more_streak", "months_paying_more"],
        ascending=False,
    ).round(4)
)

print("Results are before rebates.")
print("Consecutive months are measured within July–December only.")

Households paying more in at least one month: 55
H2 savers with at least one loss month: 48
Households paying more for at least three consecutive months: 10

Households with monthly charge increases:


,months_paying_more,months_above_threshold,months_profile_changed,months_profile_unavailable,worst_month_increase_pct,longest_paying_more_streak,h2_saving_gbp,h2_saving_pct,h2_saver_with_loss_month
household_id,,,,,,,,,
MAC004646,6,5,0,0,5.9717,6,-13.6805,-2.7651,False
MAC002806,4,2,0,0,5.2706,4,-3.1918,-1.8914,False
MAC004908,4,2,0,0,2.6911,4,-8.2497,-1.1744,False
MAC001485,3,0,0,0,1.5535,3,0.6455,0.5643,True
MAC002170,3,3,0,0,3.3001,3,-3.7675,-0.9522,False
MAC002517,3,2,0,0,3.1549,3,-0.6372,-0.2272,False
MAC003668,3,0,0,0,1.4811,3,5.7626,0.6190,True
MAC004026,3,1,0,0,2.7805,3,2.4762,0.9487,True
MAC004532,3,0,0,0,1.6847,3,-0.3429,-0.1025,False


Results are before rebates.
Consecutive months are measured within July–December only.


In [21]:
monthly_output_folder = project_folder / "reports" / "monthly_monitoring"
monthly_output_folder.mkdir(parents=True, exist_ok=True)

monthly_monitoring = monthly_monitoring.sort_values(
    ["household_id", "month_utc"]
).copy()

# Preserve the exact source run for each month's records.
monthly_monitoring["source_run_id"] = (
    monthly_monitoring["month_utc"].map({
        month: folder.name
        for month, folder in monthly_run_folders.items()
    })
)
assert monthly_monitoring["source_run_id"].notna().all()

monthly_monitoring.to_csv(
    monthly_output_folder / "customer_monthly_monitoring_2013.csv",
    index=False,
    encoding="utf-8-sig",
)
monthly_summary.to_csv(
    monthly_output_folder / "monthly_portfolio_summary_2013.csv",
    encoding="utf-8-sig",
)
household_monthly_review.to_csv(
    monthly_output_folder / "household_monthly_review_2013.csv",
    encoding="utf-8-sig",
)

findings = {
    "period": "July–December 2013, UTC interval starts",
    "households": 248,
    "households_with_any_loss_month": int(
        household_monthly_review["months_paying_more"].gt(0).sum()
    ),
    "h2_savers_with_loss_month": int(
        household_monthly_review["h2_saver_with_loss_month"].sum()
    ),
    "households_with_three_consecutive_loss_months": int(
        household_monthly_review["longest_paying_more_streak"]
        .ge(3).sum()
    ),
    "monthly_to_h2_reconciliation_passed": bool(
        differences.le(1e-6).all().all()
    ),
    "rebates_applied": False,
    "limitations": [
        "Historical backtest, not live monitoring.",
        "Monthly 2% thresholds are review flags, not bill guarantees.",
        "Profile changes do not establish tariff unsuitability.",
        "Zero-consumption profiles remain unclassified.",
        "Consecutive-month measures cover this six-month window only.",
        "No automatic switching or rebate payments were performed.",
    ],
}

(monthly_output_folder / "monthly_monitoring_findings_2013.json").write_text(
    json.dumps(findings, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Monthly customer rows saved:", len(monthly_monitoring))
print("Monthly portfolio rows saved:", len(monthly_summary))
print("Household review rows saved:", len(household_monthly_review))
print("Monthly monitoring findings saved.")
print("Output folder:", monthly_output_folder)

Monthly customer rows saved: 1488
Monthly portfolio rows saved: 6
Household review rows saved: 248
Monthly monitoring findings saved.
Output folder: D:\electricity-tariff-optimisation\reports\monthly_monitoring
